In [ ]:
# ================= 时序表型数据统计图（全局/单颗种子） =================
import json
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import defaultdict
from skimage.morphology import skeletonize

# ---------- 颜色定义（沿用原代码配色） ----------
COLOR_RED   = "#E89B9B"   # 种子
COLOR_GREEN = "#9EC29E"   # 根
COLOR_BLUE  = "#7FACCF"   # 叶
STATE_COLORS = {
    "Inert":      "#D3D3D3",
    "Germinated": "#9EC29E",
    "Shedding":   "#E89B9B",
}

# ---------- matplotlib 全局参数 ----------
plt.rcParams['font.family']        = 'Arial'
plt.rcParams['font.weight']        = 'normal'
plt.rcParams['axes.labelweight']   = 'normal'
plt.rcParams['axes.titleweight']   = 'normal'
plt.rcParams['axes.labelsize']     = 18
plt.rcParams['xtick.labelsize']    = 10
plt.rcParams['ytick.labelsize']    = 10
plt.rcParams['legend.fontsize']    = 10
plt.rcParams['lines.linewidth']    = 0.5
plt.rcParams['axes.linewidth']     = 0.5
plt.rcParams['xtick.major.width']  = 0.5
plt.rcParams['ytick.major.width']  = 0.5


# =============================================================
#  工具函数
# =============================================================

def natural_sort_key(s):
    """自然排序（1,2,...,10 而非 1,10,2,...）"""
    return [int(c) if c.isdigit() else c.lower()
            for c in re.split(r'(\d+)', str(s))]


def get_area(points):
    """Shoelace 公式计算多边形面积（pixels²）"""
    x = [p[0] for p in points]
    y = [p[1] for p in points]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))


def poly_to_mask(points, img_h, img_w):
    """将多边形 points 转为二值掩膜"""
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    pts  = np.array(points, dtype=np.int32).reshape(-1, 1, 2)
    cv2.fillPoly(mask, [pts], 1)
    return mask


def extract_hsv_mean(img_bgr, mask):
    """从 BGR 图像中提取掩膜区域的 HSV 均值；无像素时返回 (nan, nan, nan)"""
    if mask.sum() == 0:
        return np.nan, np.nan, np.nan
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    pixels  = img_hsv[mask == 1]   # shape: (N, 3)
    return float(pixels[:, 0].mean()), float(pixels[:, 1].mean()), float(pixels[:, 2].mean())


def calc_root_length(points, img_h, img_w):
    """对根多边形做骨架化，返回骨架像素数（主根长度，pixels）"""
    mask = poly_to_mask(points, img_h, img_w)
    if mask.sum() == 0:
        return 0.0
    skel = skeletonize(mask.astype(bool))
    return float(skel.sum())


# =============================================================
#  数据解析
# =============================================================

def parse_time_series(root_dir):
    """
    读取 root_dir 下所有 .json 文件及同名 .jpg 图像。
    返回：
        time_points : 文件名列表（不含后缀），按自然排序
        data        : {t_idx: {slot_id: {各表型字段}}}
    """
    root_dir   = Path(root_dir)
    json_files = sorted(root_dir.glob("*.json"), key=lambda p: natural_sort_key(p.stem))
    if not json_files:
        raise ValueError("未找到任何 JSON 文件，请检查路径。")

    time_points = [f.stem for f in json_files]
    data = {}

    for t_idx, jpath in enumerate(json_files):
        # ---------- 读取 JSON ----------
        with open(jpath, 'r', encoding='utf-8') as f:
            anno = json.load(f)
        shapes = anno.get('shapes', [])

        # ---------- 读取对应 JPG ----------
        jpg_path = jpath.with_suffix('.jpg')
        if not jpg_path.exists():
            print(f"  [警告] 未找到图像 {jpg_path.name}，颜色/骨架字段将为 NaN/0")
            img_bgr = None
            img_h, img_w = anno.get('imageHeight', 0), anno.get('imageWidth', 0)
        else:
            img_bgr = cv2.imread(str(jpg_path))
            img_h, img_w = img_bgr.shape[:2]

        # ---------- 按 group_id 聚合 ----------
        slots = defaultdict(lambda: {
            "seed_area": 0.0, "leaf_area": 0.0,   # 移除了 root_area
            "condition": "Inert",
            "seed_h": np.nan, "seed_s": np.nan, "seed_v": np.nan,
            "root_h": np.nan, "root_s": np.nan, "root_v": np.nan,
            "leaf_h": np.nan, "leaf_s": np.nan, "leaf_v": np.nan,
            "root_length": 0.0,
            "_seed_pts": None, "_root_pts": None, "_leaf_pts": None,
        })

        for shape in shapes:
            gid = shape.get('group_id')
            if gid is None:
                continue
            gid   = int(gid)
            label = shape['label'].lower()
            pts   = shape['points']
            area  = get_area(pts)

            if 'seed' in label:
                slots[gid]["seed_area"]  = area
                slots[gid]["_seed_pts"]  = pts
            elif 'root' in label:
                # 根不再保存面积，只保存长度，但保留面积字段以兼容？直接不存面积
                slots[gid]["_root_pts"]  = pts
            elif 'leaf' in label:
                slots[gid]["leaf_area"]  = area
                slots[gid]["_leaf_pts"]  = pts

            if 'condition' in shape:
                slots[gid]["condition"] = shape['condition']

        # ---------- 提取颜色 & 骨架长度 ----------
        for gid, s in slots.items():
            if img_bgr is not None:
                if s["_seed_pts"]:
                    mask = poly_to_mask(s["_seed_pts"], img_h, img_w)
                    s["seed_h"], s["seed_s"], s["seed_v"] = extract_hsv_mean(img_bgr, mask)
                if s["_root_pts"]:
                    mask = poly_to_mask(s["_root_pts"], img_h, img_w)
                    s["root_h"], s["root_s"], s["root_v"] = extract_hsv_mean(img_bgr, mask)
                    s["root_length"] = calc_root_length(s["_root_pts"], img_h, img_w)
                if s["_leaf_pts"]:
                    mask = poly_to_mask(s["_leaf_pts"], img_h, img_w)
                    s["leaf_h"], s["leaf_s"], s["leaf_v"] = extract_hsv_mean(img_bgr, mask)

            # 清理临时字段
            for k in ["_seed_pts", "_root_pts", "_leaf_pts"]:
                s.pop(k, None)

        data[t_idx] = dict(slots)

    return time_points, data


# =============================================================
#  数据总表导出
# =============================================================

def export_csv(time_points, data, out_path):
    rows = []
    for t_idx, img_name in enumerate(time_points):
        for gid, s in data[t_idx].items():
            rows.append({
                "image_name":  img_name,
                "group_id":    gid,
                "condition":   s["condition"],
                "seed_h":      s["seed_h"],
                "seed_s":      s["seed_s"],
                "seed_v":      s["seed_v"],
                "root_h":      s["root_h"],
                "root_s":      s["root_s"],
                "root_v":      s["root_v"],
                "leaf_h":      s["leaf_h"],
                "leaf_s":      s["leaf_s"],
                "leaf_v":      s["leaf_v"],
                "seed_area":   s["seed_area"],
                "leaf_area":   s["leaf_area"],
                "root_length": s["root_length"],   # 改为 root_length
            })
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"数据总表已保存至 {out_path}")


# =============================================================
#  全局模式图表
# =============================================================

def _xticks(ax, n_time):
    """所有整数位置保留刻度线，仅在5的倍数处显示数字标签。"""
    x = range(1, n_time + 1)
    labels = [str(i) if i % 5 == 0 else '' for i in x]
    plt.sca(ax)
    plt.xticks(x, labels, rotation=30, ha='right')


def plot_global_stacked(time_points, data):
    """图1：萌发进程堆叠面积图 + T₅₀标注"""
    n_time = len(time_points)
    x = np.arange(1, n_time + 1)

    count_inert = np.zeros(n_time)
    count_germ  = np.zeros(n_time)
    count_shed  = np.zeros(n_time)

    for t in range(n_time):
        for s in data[t].values():
            c = s["condition"]
            if c == "Inert":
                count_inert[t] += 1
            elif c == "Germinated":
                count_germ[t] += 1
            elif c == "Shedding":
                count_shed[t] += 1

    total = count_inert + count_germ + count_shed
    total_seeds = total[0] if total[0] > 0 else 1

    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    ax.stackplot(x,
                 count_inert, count_germ, count_shed,
                 labels=['Inert', 'Germinated', 'Shedding'],
                 colors=[STATE_COLORS["Inert"],
                         STATE_COLORS["Germinated"],
                         STATE_COLORS["Shedding"]],
                 alpha=0.85)

    # T₅₀：首次 Germinated + Shedding≥ 50% 总数的帧
    germinated_total = count_germ + count_shed
    germ_ratio = germinated_total / total_seeds
    t50_candidates = np.where(germ_ratio >= 0.5)[0]
    if len(t50_candidates) > 0:
        t50_x = t50_candidates[0] + 1
        ax.axvline(x=t50_x, color='#555555', linestyle='--', linewidth=0.8)
        ax.text(t50_x + 0.1, ax.get_ylim()[1] * 0.95,
                f'T50 = Frame {t50_x}', fontsize=8, va='top', color='#555555')

    ax.set_xlabel('Frame', fontsize=11)
    ax.set_ylabel('Number of seeds', fontsize=11)
    ax.legend(frameon=False, loc='upper left')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    _xticks(ax, n_time)
    plt.tight_layout()
    return fig


def plot_global_heatmap(time_points, data):
    """图2：群体生长速率热图（颜色编码根长度）"""
    n_time = len(time_points)

    # 收集所有出现过的 group_id，自然排序
    all_ids = sorted(
        set(gid for t in range(n_time) for gid in data[t].keys()),
        key=lambda v: natural_sort_key(str(v))
    )
    n_ids = len(all_ids)
    id_idx = {gid: i for i, gid in enumerate(all_ids)}

    matrix = np.full((n_ids, n_time), np.nan)
    for t in range(n_time):
        for gid, s in data[t].items():
            matrix[id_idx[gid], t] = s["root_length"]   # 改为 root_length

    fig, ax = plt.subplots(figsize=(max(8, n_time * 0.35), max(5, n_ids * 0.28)), dpi=300)
    im = ax.imshow(matrix, aspect='auto', cmap='YlGn',
                   interpolation='nearest', vmin=0)
    cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label('Root Length (pixels)', fontsize=18)   # 单位改为 pixels
    cbar.ax.tick_params(labelsize=15)

    ax.set_xlabel('Frame', fontsize=18)
    ax.set_ylabel('Seed ID', fontsize=18)
    ax.set_xticks(range(n_time))
    ax.set_xticklabels([str(i + 1) if (i + 1) % 5 == 0 else '' for i in range(n_time)],
                       rotation=30, ha='right', fontsize=15)
    ax.set_yticks(range(n_ids))
    ax.set_yticklabels([str(gid) for gid in all_ids], fontsize=15)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    return fig


def plot_global_mean_sd(time_points, data):
    """图3：核心表型均值±SD折线图（双Y轴：左侧面积类，右侧根长度）"""
    n_time = len(time_points)
    x = np.arange(1, n_time + 1)

    # 收集各指标均值与标准差
    seed_means = []
    seed_sds   = []
    leaf_means = []
    leaf_sds   = []
    root_means = []
    root_sds   = []

    for t in range(n_time):
        seed_vals = []
        leaf_vals = []
        root_vals = []
        for s in data[t].values():
            if s["seed_area"] > 0:
                seed_vals.append(s["seed_area"])
            if s["leaf_area"] > 0:
                leaf_vals.append(s["leaf_area"])
            if s["root_length"] > 0:
                root_vals.append(s["root_length"])
        seed_means.append(np.mean(seed_vals) if seed_vals else 0.0)
        seed_sds.append(np.std(seed_vals) if seed_vals else 0.0)
        leaf_means.append(np.mean(leaf_vals) if leaf_vals else 0.0)
        leaf_sds.append(np.std(leaf_vals) if leaf_vals else 0.0)
        root_means.append(np.mean(root_vals) if root_vals else 0.0)
        root_sds.append(np.std(root_vals) if root_vals else 0.0)

    fig, ax1 = plt.subplots(figsize=(8, 5), dpi=300)
    # 左Y轴：面积类（pixels²）
    ax1.set_xlabel('Frame', fontsize=11, color='black')
    ax1.set_ylabel('Area (pixels²)', fontsize=11, color='black')
    ax1.tick_params(axis='y', labelcolor='black')
    # 右Y轴：根长度（pixels）
    ax2 = ax1.twinx()
    ax2.set_ylabel('Root length (pixels)', fontsize=11, color='black')
    ax2.tick_params(axis='y', labelcolor='black')
    # 设置根长度y轴上限为300（可根据实际数据调整）
    ax2.set_ylim(0, 300)

    # 绘制种子面积（左轴）
    l1, = ax1.plot(x, seed_means, marker='o', color=COLOR_RED, label='Seed area mean±SD',
                   linewidth=0.5, markersize=3)
    ax1.fill_between(x, np.array(seed_means) - np.array(seed_sds),
                     np.array(seed_means) + np.array(seed_sds),
                     color=COLOR_RED, alpha=0.20, linewidth=0)

    # 绘制叶面积（左轴）
    l2, = ax1.plot(x, leaf_means, marker='^', color=COLOR_BLUE, label='Leaf area mean±SD',
                   linewidth=0.5, markersize=3)
    ax1.fill_between(x, np.array(leaf_means) - np.array(leaf_sds),
                     np.array(leaf_means) + np.array(leaf_sds),
                     color=COLOR_BLUE, alpha=0.20, linewidth=0)

    # 绘制根长度（右轴）
    l3, = ax2.plot(x, root_means, marker='s', color=COLOR_GREEN, label='Root length mean±SD',
                   linewidth=0.5, markersize=3)
    ax2.fill_between(x, np.array(root_means) - np.array(root_sds),
                     np.array(root_means) + np.array(root_sds),
                     color=COLOR_GREEN, alpha=0.20, linewidth=0)

    # 合并图例（从两个轴收集）
    lines = [l1, l2, l3]
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, frameon=False, loc='upper left', fontsize=9)

    # 设置X轴刻度
    _xticks(ax1, n_time)

    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    # 右轴保留右边框线但设为黑色（可选）
    ax2.spines['right'].set_visible(True)
    ax2.spines['right'].set_color('black')
    ax2.spines['right'].set_linewidth(0.5)

    plt.tight_layout()
    return fig


# =============================================================
#  单颗模式图表（只保留形态时序图）
# =============================================================

def plot_single(time_points, data, slot_id):
    """
    单颗种子：双轴图（左轴面积类，右轴根长度）
    右轴根长度上限 300 像素，所有文字/刻度黑色。
    状态标注修改为：只标注每个状态的第一次出现，避免重复 Shedding。
    """
    n_time = len(time_points)
    x = np.arange(1, n_time + 1)

    seed_areas   = []
    root_lengths = []
    leaf_areas   = []
    conditions   = []

    for t in range(n_time):
        slot = data[t].get(slot_id)
        if slot:
            seed_areas.append(slot["seed_area"])
            root_lengths.append(slot["root_length"])
            leaf_areas.append(slot["leaf_area"])
            conditions.append(slot["condition"])
        else:
            seed_areas.append(0.0)
            root_lengths.append(0.0)
            leaf_areas.append(0.0)
            conditions.append("Inert")

    # 状态转变帧索引（用于画竖线，仍保留）
    state_change_idx = []
    prev = conditions[0]
    for i, s in enumerate(conditions):
        if s != prev:
            state_change_idx.append(i)
            prev = s

    fig, ax1 = plt.subplots(figsize=(8, 5), dpi=300)
    fig.suptitle(f'Seed ID {slot_id} — Phenotypic Time Series',
                 fontsize=12, fontweight='normal', y=1.01)

    # 左Y轴：面积类（种子、叶片）
    ax1.set_xlabel('Frame', fontsize=11, color='black')
    ax1.set_ylabel('Area (pixels²)', fontsize=11, color='black')
    ax1.tick_params(axis='y', labelcolor='black')

    # 右Y轴：根长度（上限400像素）
    ax2 = ax1.twinx()
    ax2.set_ylabel('Root length (pixels)', fontsize=11, color='black')
    ax2.tick_params(axis='y', labelcolor='black')
    ax2.set_ylim(0, 400)

    # 绘制种子面积（左轴）
    l1, = ax1.plot(x, seed_areas, marker='o', color=COLOR_RED, label='Seed area',
                   linewidth=0.5, markersize=3)
    # 绘制叶面积（左轴）
    l2, = ax1.plot(x, leaf_areas, marker='^', color=COLOR_BLUE, label='Leaf area',
                   linewidth=0.5, markersize=3)
    # 绘制根长度（右轴）
    l3, = ax2.plot(x, root_lengths, marker='s', color=COLOR_GREEN, label='Root length',
                   linewidth=0.5, markersize=3)

    # 计算左轴最大值为标注文字提供参考（取种子和叶片的最大值）
    combined = seed_areas + leaf_areas
    max_val = max(combined) if any(v > 0 for v in combined) else 1.0

    # 绘制状态转变竖线（所有变化点）
    for idx in state_change_idx:
        ax1.axvline(x=idx + 0.5, color='gray', linestyle='--',
                    linewidth=0.5, alpha=0.7)

    # ========== 修改后的状态标注：只标注每个状态第一次出现 ==========
    first_occurrence = {}
    for i, state in enumerate(conditions):
        if state not in first_occurrence:
            first_occurrence[state] = i

    for state, idx in first_occurrence.items():
        ax1.text(idx + 1, max_val * 0.60, state,
                 fontsize=6, ha='center', va='top', rotation=45, alpha=0.8)

    # 合并图例（从两个轴收集）
    lines = [l1, l2, l3]
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, frameon=False, loc='upper left', fontsize=9)

    # 设置X轴刻度
    plt.sca(ax1)
    plt.xticks(x, [str(i) if i % 5 == 0 else '' for i in x], rotation=30, ha='right')

    # 边框样式
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_color('black')
    ax2.spines['right'].set_linewidth(0.5)

    plt.tight_layout()
    return fig


# =============================================================
#  交互式主程序
# =============================================================

def main():
    raw = input("请输入 JSON/JPG 所在文件夹路径: ").strip().strip('"').strip("'").replace('\\', '/')
    root_dir = Path(raw).resolve()
    if not root_dir.exists():
        print("文件夹不存在，请重新运行。")
        return

    print("正在加载时序数据（含图像颜色与骨架化计算，请稍候）...")
    time_points, data = parse_time_series(root_dir)
    print(f"共找到 {len(time_points)} 个时间点")

    # ---------- 导出数据总表（两种模式均导出） ----------
    csv_path = root_dir / "phenotype_data.csv"
    export_csv(time_points, data, csv_path)

    # ---------- 选择模式 ----------
    mode = input("选择模式：输入 'global' 全局模式 或 'single' 单颗种子模式: ").strip().lower()

    if mode == 'global':
        # 图1：萌发进程堆叠面积图
        fig1 = plot_global_stacked(time_points, data)
        plt.show()
        if input("是否保存萌发进程图？(y/n): ").strip().lower() == 'y':
            p = root_dir / "global_germination_stack.png"
            fig1.savefig(p, dpi=300, bbox_inches='tight')
            print(f"已保存至 {p}")
        plt.close(fig1)

        # 图2：群体生长速率热图
        fig2 = plot_global_heatmap(time_points, data)
        plt.show()
        if input("是否保存生长热图？(y/n): ").strip().lower() == 'y':
            p = root_dir / "global_growth_heatmap.png"
            fig2.savefig(p, dpi=300, bbox_inches='tight')
            print(f"已保存至 {p}")
        plt.close(fig2)

        # 图3：核心表型均值±SD
        fig3 = plot_global_mean_sd(time_points, data)
        plt.show()
        if input("是否保存均值±SD折线图？(y/n): ").strip().lower() == 'y':
            p = root_dir / "global_mean_sd.png"
            fig3.savefig(p, dpi=300, bbox_inches='tight')
            print(f"已保存至 {p}")
        plt.close(fig3)

    elif mode == 'single':
        try:
            seed_id = int(input("请输入种子 ID (0~35): ").strip())
            if seed_id not in range(36):
                print("ID 超出 0~35 范围。")
                return
        except ValueError:
            print("请输入整数。")
            return

        fig = plot_single(time_points, data, seed_id)
        plt.show()
        if input("是否保存图片？(y/n): ").strip().lower() == 'y':
            p = root_dir / f"seed_{seed_id}_phenotypic_trend.png"
            fig.savefig(p, dpi=300, bbox_inches='tight')
            print(f"已保存至 {p}")
        plt.close(fig)

    else:
        print("模式无效，请输入 'global' 或 'single'。")


if __name__ == "__main__":
    main()